In [ ]:
import os
from os.path import expanduser
home = expanduser("~/")

import sys
# sys.path.insert(0, '/global/u2/x/xshuang/gigalens-xh-dev/src')

# import sys
conda_env = sys.path[1]
del sys.path[1]

import os
# sys.path.append(f'{os.environ['HOME']}/gigalens_personal/gigalens/src')
sys.path.append(home+'/gigalens'+'/src')
sys.path.append(conda_env)
sys.path.append(home+'/GIGALens-Code/')
print(sys.path)

# import os
# os.environ["JAX_DEBUG_NANS"] = "True" 

srcdir = os.path.join(home, "gigalens/src/")


In [ ]:
import tensorflow_probability.substrates.jax as tfp

from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear

import matplotlib.patches as mpatches
from scipy.stats import norm, kstest

import jax
from jax import random
import numpy as np
import optax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import optax
import corner
import yaml
import pickle
from helpers import *
import blackjax
import importlib
tfd = tfp.distributions

import mclmc_alt
import mclmc_parallel
from mclmc_parallel import init_multi, build_kernel_multi, mclmc_multi
from mclmc_alt import isokinetic_mclachlan_smart, MCLMCAdaptationState, mclmc_find_L_and_step_size_smart,mclachlan_coefficients
import importlib
import time

In [ ]:
lens_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                theta_E=tfd.LogNormal(jnp.log(2.0), 0.5),
                gamma=tfd.TruncatedNormal(2, 0.5, 1, 3),
                e1=tfd.TruncatedNormal(0, 0.25,-0.3, 0.3),
                e2=tfd.TruncatedNormal(0, 0.25, -0.3, 0.3),
                center_x=tfd.Normal(0, 0.1),
                center_y=tfd.Normal(0, 0.1),
            )
        ),
        tfd.JointDistributionNamed(
            dict(gamma1=tfd.Normal(0, 0.1), gamma2=tfd.Normal(0, 0.1))
        ),
    ]
)
lens_light_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.05), 0.1), #sean's has 0.05 width
                n_sersic=tfd.Uniform(1, 15), #seans has 1,10
                e1=tfd.TruncatedNormal(0, 0.1, -0.5, 0.5), #sean's has -0.3, 0.3
                e2=tfd.TruncatedNormal(0, 0.1, -0.5, 0.5),
                center_x=tfd.Normal(-3.25, 0.05),
                center_y=tfd.Normal(-3.25, 0.05),
                # Ie=tfd.LogNormal(jnp.log(300.0), 3.0),
            )
        ),
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.25), 0.1),
                n_sersic=tfd.Uniform(1, 10),
                e1=tfd.TruncatedNormal(0, 0.05, -0.3, 0.3),
                e2=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
                center_x=tfd.Normal(0, 0.05),
                center_y=tfd.Normal(0, 0.05),
                # Ie=tfd.LogNormal(jnp.log(300.0), 3.0),
            )
        ),
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.25), 0.1),
                n_sersic=tfd.Uniform(1, 10),
                e1=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
                e2=tfd.TruncatedNormal(0, 0.05, -0.3, 0.3),
                center_x=tfd.Normal(0, 0.05),
                center_y=tfd.Normal(0, 0.05),
                # Ie=tfd.LogNormal(jnp.log(300.0), 3.0),
            )
        )
    ]
)

source_light_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.2), 0.5),
                n_sersic=tfd.Uniform(0.1, 15), #seans has lower bound of 0.5
                e1=tfd.TruncatedNormal(0, 0.25, -0.5, 0.5),
                e2=tfd.TruncatedNormal(0, 0.25, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.5),
                center_y=tfd.Normal(0, 0.5),
                # Ie=tfd.LogNormal(jnp.log(300.0), 3.0),
            )
        ),
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.2), 0.5),
                n_sersic=tfd.Uniform(0.1, 15),
                e1=tfd.TruncatedNormal(0, 0.25, -0.5, 0.5),
                e2=tfd.TruncatedNormal(0, 0.25, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.5),
                center_y=tfd.Normal(0, 0.5),
                # Ie=tfd.LogNormal(jnp.log(300.0), 3.0),
            )
        ),
    ]
)
prior = tfd.JointDistributionSequential(
    [lens_prior, lens_light_prior, source_light_prior]
)


In [ ]:
kernel = np.load("desi238data/psf94.npy").astype(np.float32)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=120, supersample=2, kernel=kernel)
phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=True),sersic.SersicEllipse(use_lstsq=True),sersic.SersicEllipse(use_lstsq=True)], [sersic.SersicEllipse(use_lstsq=True),sersic.SersicEllipse(use_lstsq=True)])
# phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False),sersic.SersicEllipse(use_lstsq=False),sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False),sersic.SersicEllipse(use_lstsq=False)])

lens_sim = LensSimulator(phys_model, sim_config, bs=1)
observed_img = np.load('desi238data/cutout238b.npy').astype(np.float32, copy=False)
background_rms = 0.007616264 #background_rms from photutils.background
exp_time = 1197.699462 #exp_time from header["EXPTIME"]
prob_model = BackwardProbModel(prior, jnp.array(observed_img), background_rms=background_rms, exp_time=exp_time)
# prob_model = ForwardProbModel(prior, jnp.array(observed_img), background_rms=background_rms, exp_time=exp_time)

model_seq = ModellingSequence(phys_model, prob_model, sim_config)

In [ ]:
background_rms = 0.007616264 #background_rms from photutils.background
exp_time = 1197.699462 #exp_time from header["EXPTIME"]
err_map = get_noise_image(observed_img, background_rms, exp_time)
plt.title("SNR Plot")
plt.imshow((observed_img/err_map)[:70])
plt.colorbar()
plt.show()

In [ ]:
print("Starting MAP")
start = time.perf_counter()
opt = optax.adabelief(1e-2, b1=0.95, b2=0.99, nesterov=True)
best, lps, chisq = model_seq.MAP(opt, seed=1, n_samples=500, num_steps=1000)
end = time.perf_counter()
print("MAP time taken: ", (end - start))

np.save("best.npy", best)

In [ ]:
# print("Starting SVI")
# start = time.perf_counter()
# opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
# qz, loss_hist = model_seq.SVI(map_best, opt, n_vi=1000, num_steps=1050)
# # jnp.save("svi_samples.npy", np.array(qz.sample(sample_shape=(50000), seed=jax.random.PRNGKey(0))))
# plt.plot(loss_hist)
# jnp.save("loss_hist", loss_hist)
# plt.savefig("svi.png")
# jnp.savez('qz.npz', loc=qz.loc, scale_tril=qz.scale_tril)
# end = time.perf_counter()
# print("SVI time taken: ", (end - start))

In [ ]:
map_best = jnp.load('best.npy')

# f = jnp.load('qz.npz')
# qz = tfd.MultivariateNormalTriL(loc=f['loc'], scale_tril=f['scale_tril'])

default_start = jnp.diag(jnp.ones((map_best.shape[-1],))) * 1e-3
no_SVI_qz = tfd.MultivariateNormalTriL(loc=jnp.squeeze(map_best), scale_tril=default_start)
qz = no_SVI_qz

# half_SVI_qz = tfd.MultivariateNormalTriL(loc=qz.loc,scale_tril=default_start)
# qz = half_SVI_qz
    

# hmc_samples = jnp.load('samples.npy').transpose((1, 2, 0, 3)).reshape(64, 10000, 38)
# hmc_mean = jnp.mean(hmc_samples, axis=(0,1))
# hmc_cov = jnp.cov(hmc_samples.reshape(-1,hmc_samples.shape[-1]).T)
# qz = tfd.MultivariateNormalFullCovariance(loc=hmc_mean, covariance_matrix=hmc_cov)

In [ ]:
# from blackjax.adaptation.mass_matrix import welford_algorithm, WelfordAlgorithmState
# from blackjax.adaptation.mclmc_adaptation import pytree_size, MCLMCAdaptationState, handle_nans, incremental_value_update
# from blackjax.mcmc.integrators import generalized_two_stage_integrator, format_isokinetic_state_output, ravel_pytree, _normalized_flatten_array
# from collections import namedtuple#NamedTuple


# def make_L_step_size_adaptation_test(
#     kernel,
#     dim,
#     desired_energy_var=1e-3,
#     trust_in_estimate=1.5,
#     num_effective_samples=150,
#     multi_chain=True,
#     num_chains=8,
#     mass_matrix_adapt=True,
#     continuous_adaptation=False,
# ):
#     """Adapts the stepsize and L of the MCLMC kernel. Designed for unadjusted MCLMC"""

#     decay_rate = (num_effective_samples - 1.0) / (num_effective_samples + 1.0)

#     welford_init, welford_update, welford_cov = welford_algorithm(is_diagonal_matrix=False)
#     def predictor(previous_state, params, adaptive_state, rng_key):
#         """does one step with the dynamics and updates the prediction for the optimal stepsize
#         Designed for the unadjusted MCHMC"""

#         time, x_average, step_size_max = adaptive_state

#         rng_key, nan_key = jax.random.split(rng_key)

#         # dynamics
#         next_state, info = kernel(params.inverse_mass_matrix)(
#             rng_key=rng_key,
#             state=previous_state,
#             L=params.L,
#             step_size=params.step_size,
#         )

#         # step updating
#         success, state, step_size_max, energy_change = handle_nans(
#             previous_state,
#             next_state,
#             params.step_size,
#             step_size_max,
#             info.energy_change,
#             nan_key,
#         )

#         # Warning: var = 0 if there were nans, but we will give it a very small weight
#         xi = (
#             jnp.square(energy_change) / (dim * desired_energy_var) #* Term in the sum
#         ) + 1e-8  # 1e-8 is added to avoid divergences in log xi
#         weight = jnp.exp(
#             -0.5 * jnp.square(jnp.log(xi) / (6.0 * trust_in_estimate))
#         )  # the weight reduces the impact of stepsizes which are much larger on much smaller than the desired one.

#         x_average = decay_rate * x_average + weight * (
#             xi / jnp.power(params.step_size, 6.0)
#         )
#         time = decay_rate * time + weight
#         step_size = jnp.power(
#             x_average / time, -1.0 / 6.0
#         )  # We use the Var[E] = O(eps^6) relation here.
#         step_size = (step_size < step_size_max) * step_size + (
#             step_size > step_size_max
#         ) * step_size_max  # if the proposed stepsize is above the stepsize where we have seen divergences
#         params_new = params._replace(step_size=step_size)

#         adaptive_state = (time, x_average, step_size_max)

#         return state, params_new, adaptive_state, success, (xi, weight)

#     def predictor_psmile(previous_state, params, adaptive_state, rng_key):
#         """
#         Does one step with the dynamics and updates the prediction for the optimal stepsize.
#         Designed for the unadjusted MCHMC using the pSMILE energy-variance tuner.
#         """
#         # Unpack the new pSMILE adaptive state
#         mu, sigma2, count, step_size_max = adaptive_state
    
#         rng_key, nan_key = jax.random.split(rng_key)
    
#         # Dynamics
#         next_state, info = kernel(params.inverse_mass_matrix)(
#             rng_key=rng_key,
#             state=previous_state,
#             L=params.L,
#             step_size=params.step_size,
#         )
    
#         # Step updating and NaN handling
#         success, state, step_size_max, energy_change = handle_nans(
#             previous_state,
#             next_state,
#             params.step_size,
#             step_size_max,
#             info.energy_change,
#             nan_key,
#         )

#         xi = (
#             jnp.square(energy_change) / (dim * desired_energy_var) #* Term in the sum
#         ) + 1e-8  # 1e-8 is added to avoid divergences in log xi
    
#         # --- pSMILE Adaptation Logic ---
#         # Default parameters from Section 4.2 of the SMILE paper
#         beta = 1-decay_rate#0.01          # EMA smoothing factor
#         delta = 0.1 #0.02         # Step size multiplier
#         alpha_param = 0.4 #0.1    # Adaptation probability parameter
#         eps = 1e-8           # Small constant to prevent division by zero
        
#         abs_dE = jnp.abs(energy_change)
#         count = count + 1
    
#         # 1. Exponential Moving Averages (Eq. 3)
#         mu_next = (1.0 - beta) * mu + beta * abs_dE
#         sigma2_next = (1.0 - beta) * sigma2 + beta * jnp.square(abs_dE - mu_next)
    
#         # Bias correction for the variance estimate in early tuning stages
#         bias_correction = 1.0 - jnp.power(1.0 - beta, count)
#         mu_hat = mu_next / bias_correction
#         sigma2_hat = sigma2_next / bias_correction
    
#         # 2. Gamma Distribution Moment Matching (Eq. 4)
#         shape = jnp.square(mu_hat) / (sigma2_hat + eps)
#         scale = sigma2_hat / (mu_hat + eps)
    
#         # 3. Wilson-Hilferty Transform for Quantiles (Appendix F.4)
#         def gamma_quantile(p_val):
#             z_p = norm.ppf(p_val)
#             term = 1.0 / (9.0 * shape + eps)
#             bracket = 1.0 - term + z_p * jnp.sqrt(term)
#             return scale * shape * jnp.power(jnp.maximum(bracket, 0.0), 3)
    
#         # Calculate tuning thresholds
#         tau_lo = gamma_quantile(alpha_param / 2.0)#(alpha_param / 3.0)
#         tau_hi = gamma_quantile(1.0-alpha_param / 2.0)#(1.0 - 2.0 * alpha_param / 3.0)
#         tau_guardrail = gamma_quantile(0.98) # Guardrail threshold (Eq. 5)
    
#         # 4. Adaptive Step Size Multiplier (Eq. 6)
#         current_eta = params.step_size
#         step_size = jnp.where(
#             abs_dE < tau_lo,
#             current_eta * (1.0 + delta),
#             jnp.where(
#                 abs_dE > tau_hi,
#                 current_eta * (1.0 - delta),
#                 current_eta
#             )
#         )
        
#         # Prevent step size from exceeding the historical divergence threshold
#         step_size = jnp.minimum(step_size, step_size_max)
    
#         params_new = params._replace(step_size=step_size)
        
#         # Repack the updated adaptive state
#         adaptive_state_new = (mu_next, sigma2_next, count, step_size_max)
    
#         return state, params_new, adaptive_state_new, success, (xi, 0)

#     def predictor_psmile_cont(previous_state, params, adaptive_state, rng_key):
#         """
#         Does one step with the dynamics and updates the prediction for the optimal stepsize.
#         Designed for the unadjusted MCHMC using the pSMILE energy-variance tuner.
#         """
#         # Unpack the new pSMILE adaptive state
#         mu, sigma2, count, step_size_max = adaptive_state
    
#         rng_key, nan_key = jax.random.split(rng_key)
    
#         # Dynamics
#         next_state, info = kernel(params.inverse_mass_matrix)(
#             rng_key=rng_key,
#             state=previous_state,
#             L=params.L,
#             step_size=params.step_size,
#         )
    
#         # Step updating and NaN handling
#         success, state, step_size_max, energy_change = handle_nans(
#             previous_state,
#             next_state,
#             params.step_size,
#             step_size_max,
#             info.energy_change,
#             nan_key,
#         )

#         xi = (
#             jnp.square(energy_change) / (dim * desired_energy_var) #* Term in the sum
#         ) + 1e-8  # 1e-8 is added to avoid divergences in log xi
    
#         # --- pSMILE Adaptation Logic ---
#         # Default parameters from Section 4.2 of the SMILE paper
#         beta = 1-decay_rate#0.01          # EMA smoothing factor
#         delta = 0.1 #0.02         # Step size multiplier
#         # alpha_param = 0.4 #0.1    # Adaptation probability parameter
#         eps = 1e-8           # Small constant to prevent division by zero
        
#         abs_dE = jnp.abs(energy_change)
#         count = count + 1
    
#         # 1. Exponential Moving Averages (Eq. 3)
#         mu_next = (1.0 - beta) * mu + beta * abs_dE
#         sigma2_next = (1.0 - beta) * sigma2 + beta * jnp.square(abs_dE - mu_next)
    
#         # Bias correction for the variance estimate in early tuning stages
#         bias_correction = 1.0 - jnp.power(1.0 - beta, count)
#         mu_hat = mu_next / bias_correction
#         sigma2_hat = sigma2_next / bias_correction
    
#         # 2. Gamma Distribution Moment Matching (Eq. 4)
#         shape = jnp.square(mu_hat) / (sigma2_hat + eps)
#         scale = sigma2_hat / (mu_hat + eps)
    
#         # # 3. Wilson-Hilferty Transform for Quantiles (Appendix F.4)
#         # def gamma_quantile(p_val):
#         #     z_p = norm.ppf(p_val)
#         #     term = 1.0 / (9.0 * shape + eps)
#         #     bracket = 1.0 - term + z_p * jnp.sqrt(term)
#         #     return scale * shape * jnp.power(jnp.maximum(bracket, 0.0), 3)
    
#         # # Calculate tuning thresholds
#         # tau_lo = gamma_quantile(alpha_param / 2.0)#(alpha_param / 3.0)
#         # tau_hi = gamma_quantile(1.0-alpha_param / 2.0)#(1.0 - 2.0 * alpha_param / 3.0)
#         # tau_guardrail = gamma_quantile(0.98) # Guardrail threshold (Eq. 5)
    
#         # # 4. Adaptive Step Size Multiplier (Eq. 6)
#         # current_eta = params.step_size
#         # step_size = jnp.where(
#         #     abs_dE < tau_lo,
#         #     current_eta * (1.0 + delta),
#         #     jnp.where(
#         #         abs_dE > tau_hi,
#         #         current_eta * (1.0 - delta),
#         #         current_eta
#         #     )
#         # )
#         cdf_value = jax.scipy.stats.gamma.cdf(abs_dE, shape, loc=0.0, scale=scale)
#         #* If cdf_value>0.5, lower the step size, if less, raise it
#         adjustment = (0.5-cdf_value)*delta
#         step_size = params.step_size * (1+adjustment)
        
#         # Prevent step size from exceeding the historical divergence threshold
#         step_size = jnp.minimum(step_size, step_size_max)
    
#         params_new = params._replace(step_size=step_size)
        
#         # Repack the updated adaptive state
#         adaptive_state_new = (mu_next, sigma2_next, count, step_size_max)
    
#         return state, params_new, adaptive_state_new, success, (xi, 0)

#     Hist = namedtuple("hist", ["pos", "step_size", "x_average", "xi", "step_size_max", "weight", "time", "nonan"])
#     def step(iteration_state, weight_and_key):
#         """does one step of the dynamics and updates the estimate of the optimal step size, tracking updates to the covariance"""

#         mask, rng_key, svi_inverse_mass_matrix = weight_and_key
#         state, params, adaptive_state, welford_state = iteration_state

#         state, params, adaptive_state, success, diagnostics = predictor_psmile_cont(
#             state, params, adaptive_state, rng_key
#         )

#         x = ravel_pytree(state.position)[0]

#         #! Choose how to update mass matrix, right now just tracking
#         welford_state = jax.lax.cond(mask, 
#             welford_update,
#             lambda welford_state, x : welford_state,
#             welford_state, x
#         )
#         xi = diagnostics[0]
#         weight = diagnostics[1]
#         h = Hist(pos = state.position, step_size= params.step_size, x_average=adaptive_state[1], 
#                  xi=xi, step_size_max=adaptive_state[2], weight=weight, time=adaptive_state[0], nonan=success)
#         return (state, params, adaptive_state, welford_state), h
        

#     def L_step_size_adaptation(state, params, num_steps1, rng_key):

#         welford_start = welford_init(state.position.shape[-1])

#         step_func = step_continuous_mass_matrix_adapt if continuous_adaptation else step
#         if continuous_adaptation:
#             print("USING CONTINUOUS MASS MATRIX ADAPTATION. This is mostly untested. Could screw up")
#         run_steps = lambda xs, state, params: jax.lax.scan(
#             step_func,
#             init=(
#                 state,
#                 params,
#                 (jnp.array(0.0), jnp.array(0.0), jnp.array(0, dtype=jnp.int32), jnp.inf),#(0.0, 0.0, jnp.inf),
#                 welford_start,
#             ),
#             xs=xs,
#         )

#         # we use the last num_steps2 to compute the diagonal preconditioner
#         mask = jnp.zeros(num_steps1)
#         num_steps_net = num_steps1

#         inverse_mass_matrix_tiled = jnp.tile(params.inverse_mass_matrix, (num_steps1, 1, 1))
        
#         # run the steps
#         if multi_chain:
#             run_key, final_key = jax.random.split(rng_key, 2)
#             L_step_size_adaptation_keys = jax.random.split(run_key, (num_chains, num_steps1))

#             run_steps = jax.jit(jax.vmap(run_steps, in_axes=((None, 0, None), 0, None), axis_name='chain'))
#         else:
#             #* Original behavior for only one chain
#             L_step_size_adaptation_keys = jax.random.split(
#                 rng_key, num_steps1 + 1
#             )
#             L_step_size_adaptation_keys, final_key = (
#                 L_step_size_adaptation_keys[:-1],
#                 L_step_size_adaptation_keys[-1],
#             )

#         carry, hist = run_steps(
#             (mask, L_step_size_adaptation_keys, inverse_mass_matrix_tiled), state, params
#         )
#         state, params, _, welford_state = carry

#         return state, params, hist
#     return L_step_size_adaptation

In [ ]:
# rng_key = jax.random.key(0)
# init_key, tune_key, run_key = jax.random.split(rng_key, 3)
# start_loc = jnp.squeeze(qz.sample(1, seed=init_key))

# initial_state = blackjax.mcmc.mclmc.init(
#     position=start_loc, logdensity_fn=log_prob, rng_key=init_key
# )

# starting_adapt_state = blackjax.adaptation.mclmc_adaptation.MCLMCAdaptationState(
#     jnp.sqrt(dim), jnp.sqrt(dim) * 0.25, inverse_mass_matrix=inv_mass_mat
# )

# def make_demo_kernel():
#     prior = make_default_prior()
#     gigal_dir = os.path.join(home,'gigalens/src/gigalens/')
#     kernel = np.load(gigal_dir + '/assets/psf.npy').astype(np.float32)
#     sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=2, kernel=kernel)
#     phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
#     lens_sim = LensSimulator(phys_model, sim_config, bs=1)
#     observed_img = np.load(gigal_dir + '/assets/demo.npy')
#     prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
#     model_seq = ModellingSequence(phys_model, prob_model, sim_config)
    
#     results = {}
#     results["MAP"] = MAPResults.load(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"), model_seq)
#     results["SVI"] = SVIResults.load(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"), model_seq)
    
#     def log_prob(z):
#         return prob_model.log_prob(lens_sim, z)[0]
    
#     inv_mass_mat = results['SVI'].qz.covariance()
    
#     # inv_mass_mat_true = jnp.cov(results['HMC'].HMC_samples_z.reshape(-1, 22).T)
    
#     transform = lambda state, info: state.position
    
#     kernel = lambda inverse_mass_matrix : blackjax.mcmc.mclmc.build_kernel(
#         logdensity_fn=log_prob,
#         integrator=integrator,
#         inverse_mass_matrix=inverse_mass_matrix,
#     )

#     start_loc = jnp.squeeze(results['SVI'].qz.sample(1, seed=init_key))

#     initial_state = blackjax.mcmc.mclmc.init(
#         position=start_loc, logdensity_fn=log_prob, rng_key=init_key
#     )
    
#     starting_adapt_state = blackjax.adaptation.mclmc_adaptation.MCLMCAdaptationState(
#         jnp.sqrt(dim), jnp.sqrt(dim) * 0.25, inverse_mass_matrix=inv_mass_mat
#     )
#     return kernel, initial_state, starting_adapt_state

# kernel_demo, initial_state_demo, starting_adapt_state_demo = make_demo_kernel()

In [ ]:
# desired_E_var = 5e-4
# starttime = time.perf_counter()
# state, params, hist_demo = make_L_step_size_adaptation_test(
#         kernel=kernel_demo,
#         dim=22,
#         desired_energy_var=desired_E_var,
#         trust_in_estimate=1.5,
#         num_effective_samples=100,
#         multi_chain=False,
#         # num_chains=num_chains,
#         mass_matrix_adapt=False,
#         continuous_adaptation=False,
#     )(initial_state_demo, starting_adapt_state_demo._replace(step_size=4), 1000, tune_key) #._replace(step_size=jnp.sqrt(22))
# print(f"Total time: {time.perf_counter()-starttime}")

# starttime = time.perf_counter()
# state, params, hist_238 = make_L_step_size_adaptation_test(
#         kernel=kernel,
#         dim=dim,
#         desired_energy_var=desired_E_var,
#         trust_in_estimate=1.5,
#         num_effective_samples=100,
#         multi_chain=False,
#         # num_chains=num_chains,
#         mass_matrix_adapt=False,
#         continuous_adaptation=False,
#     )(initial_state, starting_adapt_state._replace(step_size=4), 1000, tune_key) #._replace(inverse_mass_matrix=blackjax_mclmc_sampler_params.inverse_mass_matrix)
# print(f"Total time: {time.perf_counter()-starttime}")

In [ ]:
# fig, axs = plt.subplots(1,4, sharex=True)
# fig.set_size_inches(20, 5)

# def diagnost_plot(h, axs, label='', color=''):
#     ax1, ax2, ax3, ax4 = axs
#     x = np.arange(0, h.step_size.shape[0])
#     ax1.plot(x, h.step_size, label=label, color=color)
#     ax1.set_yscale('log')
#     ax1.set_ylabel("Step Size")
    
#     ax2.plot(h.x_average, label=label, color=color)
#     ax2.set_yscale('log')
#     ax2.set_ylabel("x_avg")

#     smooth_kernel_size = 30
#     kernel = np.ones(smooth_kernel_size) / smooth_kernel_size
#     xi_smoothed = np.convolve(h.xi, kernel, mode='same')
    
#     ax3.plot(h.xi, alpha=0.5, color=color)
#     ax3.plot(xi_smoothed, label=label, alpha=1.0, color=color)
#     ax3.axhline(1.0, color='black', linestyle='--')
#     ax3.set_yscale('log')
#     ax3.set_ylim(bottom=1e-4)
#     ax3.set_ylabel('xi')

#     # xi_weight = h.xi * h.weight/np.pow(step_size, 6)
#     # xi_weight_smoothed = np.convolve(xi_weight, kernel, mode='same')
#     ax4.plot(h.nonan, alpha=0.5, color=color)
#     # ax4.plot(xi_weight_smoothed, label=label, alpha=1.0, color=color)
#     # ax4.set_yscale('log')
#     ax4.set_ylabel('NaNs?')
#     # if jnp.any(h.
#     print(jnp.mean(h.xi))
    

# diagnost_plot(hist_demo, axs, label='Demo', color='C0')
# diagnost_plot(hist_238, axs, label='DESI238', color='C1')

# for ax in axs:
#     ax.set_xlabel("Step")
#     ax.legend()
# plt.suptitle(f"Comparison in Step Size Adaptation with desired_E_var: {desired_E_var:.1e}")
# plt.show()

In [ ]:
import alternate_inference.mclmc_alt
importlib.reload(alternate_inference.mclmc_alt)
from alternate_inference.mclmc_alt import MCLMC_JIT, isokinetic_velocity_verlet_smart

num_burnin_steps = 2000
num_results=2000
frac_tune1=0.2 #* initial step size tuning
frac_tune2=0.6 #* Used for mass matrix adaptation
frac_tune3=0.2 #! Tuning L. ~10 effective samples are needed for this to be accurate

debug_hist = MCLMC_JIT(
    model_seq, qz, 
    n_hmc=8, num_burnin_steps=num_burnin_steps, num_results=num_results, 
    desired_energy_variance=5e-4, frac_tune1=frac_tune1, frac_tune2=frac_tune2, frac_tune3=frac_tune3,
    seed=0, debug_output=True, step_size_adapt_use_psmile=False, use_shard_map=True,progress_bar=True, windowed_mass_matrix=True,
    # integrator=isokinetic_velocity_verlet_smart
)
mclmc_samples = debug_hist.position[:, -num_results:, :]

In [ ]:
multi_chain_samples = mclmc_samples

In [ ]:
print(debug_hist.step_size[0, -1])
stage1 = int(frac_tune1*num_burnin_steps)
stage2 = int((frac_tune1+frac_tune2)*num_burnin_steps)
stage3 = int((frac_tune1+frac_tune2+frac_tune3)*num_burnin_steps)

fig, axs = plt.subplots(5, 1, sharex=True)
ax1, ax2, ax3, ax4, ax_last =axs
fig.set_size_inches(10, 8)
ax1.plot(debug_hist.step_size.T)
ax1.set_title("Chain-Wise Step Size")
ax1.set_ylabel("Step Size")
# ax1.set_ylim(top=10)
# ax1.set_yscale('log')

ax2.plot(debug_hist.L.T)
ax2.set_title("Chain-Wise L")
ax2.set_ylabel("L")
# ax2.set_ylim(top=20)



inverse_mass_matrix_hist = debug_hist.inverse_mass_matrix[0]
vmapped_eigval = jax.vmap(lambda x: jnp.linalg.eig(x)[0])
mass_mat_eigval = vmapped_eigval(inverse_mass_matrix_hist)
min_eigval = jnp.min(mass_mat_eigval, axis=1)
max_eigval = jnp.max(mass_mat_eigval, axis=1)
mean_eigval = jnp.mean(mass_mat_eigval, axis=1)

ax3.plot(min_eigval, label='Min', color='blue')
ax3.plot(max_eigval, label='Max', color='red')
ax3.plot(mean_eigval, label='Mean', color='black')
ax3.legend()
ax3.set_title("Covariance Eigenvalues")
ax3.set_yscale('log')
ax3.set_ylabel("Eigenvalue")



smooth_kernel_size = 30
kernel = np.ones(smooth_kernel_size) / smooth_kernel_size
xi_chain = debug_hist.xi[8]
xi_smoothed = np.convolve(xi_chain, kernel, mode='same')
ax4.plot(xi_chain, alpha=0.5, color='blue')
ax4.plot(xi_smoothed, alpha=1.0, color='blue')
ax4.set_yscale('log')
ax4.set_ylabel("xi for chain 0")
ax4.axhline(1.0, color='black', linestyle='--')

ax_last.set_xlabel("Step")

ax_last.set_title("Nans?")
ax_last.imshow(debug_hist.nonan[:,:stage2], aspect='auto', interpolation='none', cmap='RdYlGn')

for ax in axs:
    ax.axvline(stage1, color='red', linestyle='--')
    ax.axvline(stage2, color='blue', linestyle='--')
    ax.axvline(stage3, color='green', linestyle='--')

    # ax.set_xlim(right=stage3)


plt.show()

In [ ]:
# n_grad_evals_mclmc = num_steps
n_grad_evals_mclmc = multi_chain_samples.shape[0]*multi_chain_samples.shape[1]
# ESS_mclmc = blackjax.diagnostics.effective_sample_size(samples_mclmc[np.newaxis, :,:], chain_axis=0, sample_axis=1)
ESS_mclmc = blackjax.diagnostics.effective_sample_size(multi_chain_samples, chain_axis=0, sample_axis=1)

treedef = jax.tree.structure(prob_model.bij.forward(list(map_best.T)))

ESS_mclmc_tree = jax.tree.unflatten(treedef, ESS_mclmc)
print(np.mean(ESS_mclmc)/n_grad_evals_mclmc, " | ", jnp.min(ESS_mclmc)/n_grad_evals_mclmc)
ESS_mclmc_tree

In [ ]:
rhat = blackjax.diagnostics.potential_scale_reduction(multi_chain_samples, chain_axis=0, sample_axis=1)
rhat_tree = jax.tree.unflatten(treedef, rhat)
print(jnp.max(rhat))
rhat_tree


In [ ]:

sample_length = list(range(20, multi_chain_samples.shape[1], 300)) + [multi_chain_samples.shape[1]]
rhats = -np.ones((len(sample_length), multi_chain_samples.shape[-1]))
for i, l in enumerate(sample_length):
    rhats[i] = blackjax.diagnostics.potential_scale_reduction(multi_chain_samples[:, :l, :], chain_axis=0, sample_axis=1)
plt.axhline(1e-2, linestyle='--', label='Convergence')
plt.plot(sample_length, rhats-1)
plt.yscale('log')
plt.ylabel("Rhat-1")
plt.xlabel("num_results")
plt.legend()
plt.ylim(bottom=1e-5)
plt.show()

In [ ]:
hmc_samp = hmc_samples[:16]
sample_length = list(range(20, hmc_samp.shape[1], 300)) + [hmc_samp.shape[1]]
rhats = -np.ones((len(sample_length), multi_chain_samples.shape[-1]))
for i, l in enumerate(sample_length):
    rhats[i] = blackjax.diagnostics.potential_scale_reduction(hmc_samp[:, :l, :], chain_axis=0, sample_axis=1)
plt.axhline(1e-2, linestyle='--', label='Convergence')
plt.plot(sample_length, rhats-1)
plt.yscale('log')
plt.ylabel("Rhat-1")
plt.xlabel("num_results")
plt.legend()
plt.show()

In [ ]:
hmc_ESS = blackjax.diagnostics.effective_sample_size(hmc_samples[:16], chain_axis=0, sample_axis=1)
np.min(hmc_ESS)

In [ ]:
print(f"MAP Log Prob: {log_prob(jnp.squeeze(map_best))}")
print(f"qz Mean Log Prob: {log_prob(qz.loc)}")
print(f"HMC Median Log Prob: {log_prob(jnp.median(multi_chain_samples, axis=(0, 1)))}")

In [ ]:
samples = multi_chain_samples
run_key = jax.random.key(0)
samples = samples.reshape(-1, samples.shape[-1]) #samples_mclmc 
MCMC_x = prob_model.bij.forward(list(samples.T))

# HMC_x = prob_model.bij.forward(list(hmc_samples.reshape(-1, hmc_samples.shape[-1]).T))

adapt_cov = debug_hist.inverse_mass_matrix[0, -1]
adapt_qz = tfd.MultivariateNormalFullCovariance(loc=jnp.mean(samples, axis=0), covariance_matrix=adapt_cov)
adapt_qz_samples = adapt_qz.sample((1000,), run_key)
adapt_qz_x = prob_model.bij.forward(list(adapt_qz_samples.T))

svi_samples = qz.sample((1000,), run_key)
svi_x = prob_model.bij.forward(list(svi_samples.T))

map_x = prob_model.bij.forward(list(map_best.T))

# HMC_1chain_x = prob_model.bij.forward(list(results['HMC'].HMC_samples_z[0, 11, -40000:].T))

plot_params = cornerplot_labels(MCMC_x)#[:20] #! Only plot some params for speed. CHANGE LATER

# n_samp = HMC_x[0][0]['e1'].shape[0]
# rand_idx = np.random.choice(np.arange(n_samp), size=(20000,), replace=False)

# fig = cornerplot_posterior(jax.tree.map(lambda x: x[rand_idx], HMC_x), color='black',plot_params=plot_params)

n_samp_MCMC = MCMC_x[0][0]['e1'].shape[0]
rand_idx_MCMC = np.random.choice(np.arange(n_samp_MCMC), size=(20000,), replace=True)
fig = cornerplot_posterior(jax.tree.map(lambda x: x[rand_idx_MCMC], MCMC_x), color='red',plot_params=plot_params)#, fig=fig)

# cornerplot_posterior(svi_x, color='blue', plot_params=plot_params, fig=fig, overplots=map_x, overplot_color='green')
cornerplot_posterior(adapt_qz_x, plot_params=plot_params, fig=fig, color='purple')
# cornerplot_posterior(mclmc_results_100sys.HMC_samples, fig=fig, color='purple', plot_params=plot_params)
# cornerplot_posterior(HMC_1chain_x, fig=fig, color='green', plot_params=plot_params)
plt.show()

In [ ]:
def bridge_sampler(rng_key, samples, log_posterior_fn, n_iter=50):
    """
    Estimates the log-marginal likelihood using the Meng-Wong bridge sampler.
    
    Args:
        rng_key: JAX random key.
        samples: MCMC samples of shape (n_samples, n_dim).
        log_posterior_fn: Function mapping (theta) -> log_posterior(theta).
        n_iter: Number of iterations for the Meng-Wong algorithm.
    """
    n_samples, n_dim = samples.shape
    
    # 1. Fit a Gaussian proposal to the MCMC samples
    mu = jnp.mean(samples, axis=0)
    cov = jnp.cov(samples, rowvar=False)
    proposal = tfd.MultivariateNormalFullCovariance(loc=mu, covariance_matrix=cov)
    
    # 2. Draw 'fake' samples from the proposal
    prop_key, _ = jax.random.split(rng_key)
    gen_samples = proposal.sample(n_samples, seed=prop_key)
    
    # 3. Evaluate log-densities (vmap for speed)
    # l1: log-posterior of MCMC samples
    # l2: log-posterior of Gen samples
    l1 = jax.vmap(log_posterior_fn)(samples)
    l2 = jax.vmap(log_posterior_fn)(gen_samples)
    # max_lp = jnp.max(l1)
    # l1 -= max_lp
    # l2 -= max_lp

    # print(l1)
    # print(l2)
    
    # q1: log-proposal of MCMC samples
    # q2: log-proposal of Gen samples
    q1 = proposal.log_prob(samples)
    q2 = proposal.log_prob(gen_samples)
    # print(q1)
    # print(q2)
    
    # 4. Iterative Meng-Wong Algorithm
    # Initial guess for the marginal likelihood (r = Z_posterior / Z_proposal)
    # Since Z_proposal = 1 (normalized), r is the evidence Z.
    log_r_guess = jnp.log(1/n_samples) + jax.nn.logsumexp(l2-q2)
    print(log_r_guess)
    log_r = log_r_guess

    
    h = []
    s1 = 0.5 # Proportion of samples from posterior (assuming n1 == n2)
    s2 = 0.5 # Proportion of samples from proposal
    
    for _ in range(n_iter):
        # We want to compute the update: 
        # r_new = (sum(q2 / (s1*q2 + s2*r*g2))) / (sum(g1 / (s1*q1 + s2*r*g1)))
        
        # Let's define the terms for the numerator and denominator in log-space
        # Numerator term: log( q(theta_gen) / (s1*q(theta_gen) + s2*r*g(theta_gen)) )
        l_num = l2 - jnp.logaddexp(jnp.log(s1) + l2, jnp.log(s2) + log_r + q2)
        
        # Denominator term: log( g(theta_mcmc) / (s1*q(theta_mcmc) + s2*r*g(theta_mcmc)) )
        l_den = q1 - jnp.logaddexp(jnp.log(s1) + l1, jnp.log(s2) + log_r + q1)
        
        # Update log_r
        log_r_new = jax.nn.logsumexp(l_num) - jax.nn.logsumexp(l_den)
        
        # For stability, we can track the delta or use a small dampening factor if needed
        log_r = log_r_new
        h.append(log_r)

    return log_r, h


In [ ]:
smp = multi_chain_samples.reshape(-1, dim)
idxes = np.random.choice(smp.shape[0], size=(200,), replace=False)
r, hist = bridge_sampler(jax.random.key(0), smp[idxes], log_prob,n_iter=50)

In [ ]:
print(r)

In [ ]:
#log(r) for two-source model: 44271.53 (200 smp) (or 44272.633 for 1000 samples)
#log(r) for two-source model: 45779.387 (200 smp) (or 45779.97 for 1000 samples)
#log(r) for three-source model: 45923.23 (200 smp) (or 45925.473 for 1000 samples)

In [ ]:
plt.plot(hist)